In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("All libraries imported successfully!")

In [ ]:
df = pd.read_csv(
    "data/data.csv",
    encoding="ISO-8859-1"
)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

In [ ]:
print("DATASET INFORMATION")
print("-" * 40)

df.info()

print("\nMISSING VALUES")
print("-" * 40)

print(df.isnull().sum())

print("\nUNIQUE CUSTOMERS:",
      df["CustomerID"].nunique())

print("\nUNIQUE PRODUCTS:",
      df["Description"].nunique())

print("\nSTATISTICAL SUMMARY")

display(df.describe())

In [ ]:
# Preserve original dataset
df_clean = df.copy()

# Remove rows with missing product descriptions
df_clean = df_clean.dropna(subset=["Description"])

# Convert CustomerID to nullable integer
df_clean["CustomerID"] = df_clean["CustomerID"].astype("Int64")

# Convert InvoiceDate to datetime
df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"]
)

print("Missing values after cleaning:")
print(df_clean.isnull().sum())

print("\nDataset shape:", df_clean.shape)

In [ ]:
# Identify cancelled invoices
cancelled = df_clean[
    df_clean["InvoiceNo"].str.startswith("C", na=False)
]

print("Cancelled transaction records:", len(cancelled))

print("Total cancelled invoices:",
      cancelled["InvoiceNo"].nunique())

print("Negative quantity records:",
      (df_clean["Quantity"] < 0).sum())

In [ ]:
sales_df = df_clean[
    (~df_clean["InvoiceNo"].str.startswith("C", na=False)) &
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0)
].copy()

# Calculate revenue
sales_df["Revenue"] = (
    sales_df["Quantity"] * sales_df["UnitPrice"]
)

print("Clean sales dataset shape:", sales_df.shape)

print("Total sales revenue:",
      round(sales_df["Revenue"].sum(), 2))

print("Unique customers:",
      sales_df["CustomerID"].nunique())

In [ ]:
print("Remaining invalid quantities:",
      (sales_df["Quantity"] <= 0).sum())

print("Remaining invalid prices:",
      (sales_df["UnitPrice"] <= 0).sum())

print("Remaining cancelled invoices:",
      sales_df["InvoiceNo"].str.startswith("C", na=False).sum())

print("\nFinal dataset information:")
sales_df.info()

In [ ]:
# Group products and calculate total quantity sold

top_products = (
    sales_df.groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("TOP 10 MOST FREQUENTLY PURCHASED PRODUCTS")
display(top_products.reset_index())

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    x=top_products.values,
    y=top_products.index
)

plt.title("Top 10 Most Frequently Purchased Products")
plt.xlabel("Total Quantity Sold")
plt.ylabel("Product Name")

plt.tight_layout()
plt.show()

In [ ]:
sales_df[
    sales_df["Description"] == "PAPER CRAFT , LITTLE BIRDIE"
].sort_values("Quantity", ascending=False).head(5)

In [ ]:
sales_df[
    sales_df["Description"] == "WORLD WAR 2 GLIDERS ASSTD DESIGNS"
].sort_values("Quantity", ascending=False).head(5)

In [ ]:
# Exclude transactions without CustomerID
customer_sales = sales_df.dropna(subset=["CustomerID"])

# Calculate total spending per customer
top_customers = (
    customer_sales.groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("TOP 10 HIGHEST-SPENDING CUSTOMERS")

display(
    top_customers.reset_index()
)

In [ ]:
plt.figure(figsize=(11, 6))

sns.barplot(
    x=top_customers.values,
    y=top_customers.index.astype(str)
)

plt.title("Top 10 Customers by Total Purchase Value")
plt.xlabel("Total Purchase Value (£)")
plt.ylabel("Customer ID")

plt.tight_layout()
plt.show()

In [ ]:
total_revenue = customer_sales["Revenue"].sum()

top_10_revenue = top_customers.sum()

contribution = (top_10_revenue / total_revenue) * 100

print("Total revenue from identified customers:",
      round(total_revenue, 2))

print("Revenue from top 10 customers:",
      round(top_10_revenue, 2))

print("Top 10 customer revenue contribution:",
      round(contribution, 2), "%")

In [ ]:
# Extract year and month
sales_df["Year"] = sales_df["InvoiceDate"].dt.year

sales_df["Month"] = sales_df["InvoiceDate"].dt.month

sales_df["YearMonth"] = sales_df["InvoiceDate"].dt.to_period("M")

print("Date range:")
print(sales_df["InvoiceDate"].min())
print(sales_df["InvoiceDate"].max())

In [ ]:
monthly_sales = (
    sales_df.groupby("YearMonth")["Revenue"]
    .sum()
    .reset_index()
)

monthly_sales["YearMonth"] = (
    monthly_sales["YearMonth"].astype(str)
)

print("MONTHLY SALES REVENUE")
display(monthly_sales)

# Identify peak and lowest sales months
print("\nHighest sales month:")
display(monthly_sales.loc[monthly_sales["Revenue"].idxmax()])

print("\nLowest sales month:")
display(monthly_sales.loc[monthly_sales["Revenue"].idxmin()])

In [ ]:
plt.figure(figsize=(13, 6))

sns.lineplot(
    data=monthly_sales,
    x="YearMonth",
    y="Revenue",
    marker="o",
    linewidth=2
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
yearly_sales = (
    sales_df.groupby("Year")["Revenue"]
    .sum()
    .reset_index()
)

print("YEARLY SALES REVENUE")
display(yearly_sales)

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=yearly_sales,
    x="Year",
    y="Revenue"
)

plt.title("Yearly Revenue Comparison")
plt.xlabel("Year")
plt.ylabel("Revenue (£)")

plt.tight_layout()
plt.show()

In [ ]:
from itertools import combinations
from collections import Counter

In [ ]:
# Group products belonging to the same invoice

invoice_products = (
    sales_df.groupby("InvoiceNo")["Description"]
    .apply(lambda x: list(set(x)))
)

print("Total invoices:", len(invoice_products))

print("\nSample invoice products:")
display(invoice_products.head())

In [ ]:
# Count product combinations
product_pairs = Counter()

for products in invoice_products:

    # Generate all possible pairs
    if len(products) >= 2:
        pairs = combinations(sorted(products), 2)

        product_pairs.update(pairs)

# Get top 10 product pairs
top_pairs = product_pairs.most_common(10)

# Convert results into DataFrame
top_pairs_df = pd.DataFrame(
    top_pairs,
    columns=["Product Pair", "Frequency"]
)

top_pairs_df["Product 1"] = top_pairs_df["Product Pair"].apply(
    lambda x: x[0]
)

top_pairs_df["Product 2"] = top_pairs_df["Product Pair"].apply(
    lambda x: x[1]
)

top_pairs_df = top_pairs_df[
    ["Product 1", "Product 2", "Frequency"]
]

print("TOP 10 FREQUENTLY PURCHASED PRODUCT PAIRS")

display(top_pairs_df)

In [ ]:
top_pairs_df["Pair"] = (
    top_pairs_df["Product 1"] +
    " + " +
    top_pairs_df["Product 2"]
)

plt.figure(figsize=(12, 7))

sns.barplot(
    data=top_pairs_df,
    x="Frequency",
    y="Pair"
)

plt.title("Top 10 Frequently Purchased Product Combinations")
plt.xlabel("Number of Transactions")
plt.ylabel("Product Combinations")

plt.tight_layout()
plt.show()

In [ ]:
# Consider the day after the last transaction as the reference date
reference_date = sales_df["InvoiceDate"].max() + pd.Timedelta(days=1)

# Remove customers with missing IDs
rfm_data = sales_df.dropna(subset=["CustomerID"])

# Calculate RFM metrics
rfm = rfm_data.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (reference_date - x.max()).days,
    "InvoiceNo": "nunique",
    "Revenue": "sum"
})

# Rename columns
rfm.columns = ["Recency", "Frequency", "Monetary"]

print("RFM CUSTOMER DATA")
display(rfm.head())

print("\nTotal customers analysed:", len(rfm))

In [ ]:
# Assign RFM scores using quintiles

rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Calculate overall RFM score
rfm["RFM_Score"] = (
    rfm["R_Score"] +
    rfm["F_Score"] +
    rfm["M_Score"]
)

display(rfm.head(10))

In [ ]:
def segment_customer(score):

    if score >= 13:
        return "Champions"

    elif score >= 10:
        return "Loyal Customers"

    elif score >= 7:
        return "At Risk"

    else:
        return "Others"


rfm["Segment"] = rfm["RFM_Score"].apply(segment_customer)

print("CUSTOMER SEGMENTATION")
display(
    rfm["Segment"].value_counts().reset_index()
)

In [ ]:
segment_counts = rfm["Segment"].value_counts()

plt.figure(figsize=(9, 5))

sns.barplot(
    x=segment_counts.index,
    y=segment_counts.values
)

plt.title("Customer Segmentation Using RFM Analysis")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
segment_revenue = (
    rfm.groupby("Segment")["Monetary"]
    .sum()
    .sort_values(ascending=False)
)

print("REVENUE CONTRIBUTION BY CUSTOMER SEGMENT")

display(segment_revenue.reset_index())

In [ ]:
country_sales = (
    sales_df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

top_countries = country_sales.head(10)

print("TOP 10 COUNTRIES BY REVENUE")

display(
    top_countries.reset_index()
)

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    x=top_countries.values,
    y=top_countries.index
)

plt.title("Top 10 Countries by Revenue")
plt.xlabel("Total Revenue (£)")
plt.ylabel("Country")

plt.tight_layout()
plt.show()

In [ ]:
total_country_revenue = country_sales.sum()

country_contribution = (
    top_countries / total_country_revenue
) * 100

print("REVENUE CONTRIBUTION BY TOP 10 COUNTRIES")

display(
    country_contribution.reset_index(
        name="Revenue Contribution (%)"
    )
)

print(
    "\nTop 10 countries contribute:",
    round(country_contribution.sum(), 2),
    "% of total revenue"
)

In [ ]:
# Exclude customers without CustomerID
customer_data = sales_df.dropna(subset=["CustomerID"])

# Count unique invoices per customer
purchase_frequency = (
    customer_data.groupby("CustomerID")["InvoiceNo"]
    .nunique()
)

# Classify customers
customer_type = purchase_frequency.apply(
    lambda x: "Repeat Customer" if x > 1 else "One-Time Customer"
)

print("CUSTOMER PURCHASE FREQUENCY")

display(
    customer_type.value_counts().reset_index()
)

In [ ]:
customer_counts = customer_type.value_counts()

total_customers = customer_counts.sum()

repeat_percentage = (
    customer_counts.get("Repeat Customer", 0)
    / total_customers
) * 100

one_time_percentage = (
    customer_counts.get("One-Time Customer", 0)
    / total_customers
) * 100

print("Total Customers:", total_customers)

print("Repeat Customers:",
      customer_counts.get("Repeat Customer", 0))

print("One-Time Customers:",
      customer_counts.get("One-Time Customer", 0))

print("Repeat Customer Percentage:",
      round(repeat_percentage, 2), "%")

print("One-Time Customer Percentage:",
      round(one_time_percentage, 2), "%")

In [ ]:
plt.figure(figsize=(7, 5))

plt.pie(
    customer_counts.values,
    labels=customer_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Repeat vs One-Time Customers")

plt.tight_layout()
plt.show()

In [ ]:
# Add customer type to transaction records
customer_revenue = customer_data.copy()

customer_revenue["CustomerType"] = (
    customer_revenue["CustomerID"].map(customer_type)
)

# Calculate revenue by customer type
revenue_by_type = (
    customer_revenue.groupby("CustomerType")["Revenue"]
    .sum()
)

print("REVENUE BY CUSTOMER TYPE")

display(revenue_by_type.reset_index())

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    x=revenue_by_type.index,
    y=revenue_by_type.values
)

plt.title("Revenue: Repeat vs One-Time Customers")
plt.xlabel("Customer Type")
plt.ylabel("Revenue (£)")

plt.tight_layout()
plt.show()

In [ ]:
# Identify cancelled invoices
cancelled_df = df_clean[
    df_clean["InvoiceNo"].str.startswith("C", na=False)
].copy()

print("CANCELLED TRANSACTION ANALYSIS")

print("Cancelled transaction records:",
      len(cancelled_df))

print("Unique cancelled invoices:",
      cancelled_df["InvoiceNo"].nunique())

# Calculate cancellation value
cancelled_df["CancellationValue"] = (
    cancelled_df["Quantity"] * cancelled_df["UnitPrice"]
)

print("Total cancellation value (£):",
      round(abs(cancelled_df["CancellationValue"].sum()), 2))

In [ ]:
# Identify negative quantity transactions
returns_df = df_clean[
    (df_clean["Quantity"] < 0) &
    (~df_clean["InvoiceNo"].str.startswith("C", na=False))
].copy()

print("RETURN ANALYSIS")

print("Return transaction records:", len(returns_df))

print("Unique return invoices:",
      returns_df["InvoiceNo"].nunique())

print("Total returned quantity:",
      abs(returns_df["Quantity"].sum()))

print("Return value (£):",
      round(abs(
          (returns_df["Quantity"] * returns_df["UnitPrice"]).sum()
      ), 2))

In [ ]:
top_returns = (
    returns_df.groupby("Description")["Quantity"]
    .sum()
    .abs()
    .sort_values(ascending=False)
    .head(10)
)

print("TOP 10 MOST RETURNED PRODUCTS")

display(top_returns.reset_index())

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    x=top_returns.values,
    y=top_returns.index
)

plt.title("Top 10 Products by Returned Quantity")
plt.xlabel("Returned Quantity")
plt.ylabel("Product")

plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Cancelled transaction records",
        "Unique cancelled invoices",
        "Returned transaction records",
        "Unique return invoices",
        "Total returned quantity",
        "Total cancellation value (£)",
        "Total return value (£)"
    ],
    "Value": [
        len(cancelled_df),
        cancelled_df["InvoiceNo"].nunique(),
        len(returns_df),
        returns_df["InvoiceNo"].nunique(),
        abs(returns_df["Quantity"].sum()),
        round(abs(cancelled_df["CancellationValue"].sum()), 2),
        round(abs(
            (returns_df["Quantity"] * returns_df["UnitPrice"]).sum()
        ), 2)
    ]
})

display(summary)

In [ ]:
print("UNIT PRICE DISTRIBUTION")
display(returns_df["UnitPrice"].value_counts().head(10))

print("\nSAMPLE ADJUSTMENT RECORDS")

display(
    returns_df[
        ["InvoiceNo", "StockCode", "Description",
         "Quantity", "UnitPrice", "CustomerID"]
    ].head(15)
)

print("\nUNIQUE DESCRIPTIONS:")
print(returns_df["Description"].nunique())

In [ ]:
# Operational adjustments
adjustments_df = returns_df.copy()

# Final cancellation and adjustment summary
final_summary = pd.DataFrame({
    "Category": [
        "Cancelled Transactions",
        "Operational Stock Adjustments"
    ],
    "Records": [
        len(cancelled_df),
        len(adjustments_df)
    ],
    "Invoices": [
        cancelled_df["InvoiceNo"].nunique(),
        adjustments_df["InvoiceNo"].nunique()
    ],
    "Quantity": [
        abs(cancelled_df["Quantity"].sum()),
        abs(adjustments_df["Quantity"].sum())
    ],
    "Value (£)": [
        round(abs(cancelled_df["CancellationValue"].sum()), 2),
        round(abs(
            (adjustments_df["Quantity"] *
             adjustments_df["UnitPrice"]).sum()
        ), 2)
    ]
})

display(final_summary)